In [ ]:
import os
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models.variant_scorers import CenterMaskScorer
from alphagenome.models.variant_scorers import AggregationType
from alphagenome.models import variant_scorers
from alphagenome.models import dna_client
import numpy as np
import pandas as pd
import jax
import polars as pl
os.environ['CUDA_VISIBLE_DEVICES'] = '6'

In [22]:
jax.devices()

[CudaDevice(id=0)]

In [ ]:
class AlphaGenomeScoreVariant:
    
    enformer_locus_to_ontology = {
        "F9": ["EFO:0001187"],
        "GP1BA": ["EFO:0002067"],
        "HBB": ["EFO:0002067"],
        "HBG1": ["EFO:0002067"],
        "HNF4A": ["UBERON:0002369"],
        "IRF4": ["CL:2000000", "CL:2000045", "EFO:0005720"],
        "IRF6": ["CL:0000312", "CL:1001606"],
        "LDLR": ["EFO:0001187"],
        "MSMB": ["UBERON:0002369"],
        "MYC": ["UBERON:0002369"],
        "PKLR": ["EFO:0002067"],
        "SORT1": ["EFO:0001187"],
        "TERT": ["UBERON:0002369"],
        "ZFAND3": ["CL:0002351", "UBERON:0001150", "UBERON:0001264"],
    }

    borzoi_locus_to_ontology = {
        "F9": ["EFO:0002067"],
        "GP1BA": ["EFO:0002067"],
        "HBB": ["EFO:0002067"],
        "HBG1": ["EFO:0002067"],
        "HNF4A": ["UBERON:0002113"],
        "IRF4": ["CL:2000000", "CL:2000045", "EFO:0005720"],
        "IRF6": ["CL:0000312", "CL:1001606"],
        "LDLR": ["UBERON:0002113"],
        "MSMB": ["UBERON:0002113"],
        "MYC": ["UBERON:0002113"],
        "PKLR": ["EFO:0002067"],
        "SORT1": ["EFO:0001187"],
        "TERT": ["UBERON:0002113"],
        "ZFAND3": ["CL:0002351", "UBERON:0001150", "UBERON:0001264"],
    }
    
    def __init__(self, model: dna_model.AlphaGenomeModel, ontology_to_use:str):
        self.model = model
        self.ontology_to_use = AlphaGenomeScoreVariant.borzoi_locus_to_ontology if ontology_to_use == 'borzoi' else AlphaGenomeScoreVariant.enformer_locus_to_ontology
        
        self.center_DNase_scorer = CenterMaskScorer(requested_output=dna_model.OutputType.DNASE, width=501, aggregation_type=AggregationType.DIFF_SUM)
        self.center_CAGE_scorer = CenterMaskScorer(requested_output=dna_model.OutputType.CAGE, width=501, aggregation_type=AggregationType.DIFF_SUM)
    
    def __call__(self, variantStrings:list, observed_change:list, elements:list):
        
        elements_clean = [el.split(' ')[0].split('.')[0].split('-')[0].split('rs')[0] for el in elements]
        ontologies = [self.ontology_to_use[el] for el in elements_clean]
        
        intervals = []
        variants = []
        for var in variantStrings:
            variant = genome.Variant.from_str(var)
            interval = variant.reference_interval.resize(2**20)
            intervals.append(interval)
            variants.append(variant)
        
        prediction = self.model.score_variants(
            intervals,
            variants,
            variant_scorers=[self.center_DNase_scorer],
            organism=dna_model.Organism.HOMO_SAPIENS,
            max_workers=20
            )
        df = variant_scorers.tidy_scores(prediction)[['variant_id', 'ontology_curie', 'raw_score']]
        df.variant_id = df.variant_id.astype(str)
        def select_by_ontology(struct, ontologies:set|list=["CL:0000312", "CL:1001606"]):
            ont_list = struct['ontology_curie']
            raw_scores = struct['raw_score']
            indices = [ont_list.index(ont) for ont in ontologies]
            raw_values = [raw_scores[idx] for idx in indices]
            return np.mean(raw_values) 
        scores = pl.DataFrame(df).group_by('variant_id') \
                        .agg(pl.col("*")) \
                        .with_columns(raw_score_averaged = pl.struct(pl.col.ontology_curie, pl.col.raw_score) \
                        .map_elements(select_by_ontology, return_dtype=pl.Float64, strategy='threading')) \
                        .sort(pl.col('variant_id').sort_by(pl.Series(variantStrings)))['raw_score_averaged'].to_list()
        
        
        return pl.DataFrame({'variants': variantStrings, 'Predicted': scores, 'Observed': observed_change})

In [124]:
dataset = pl.read_csv('/home/jovyan/.cache/mpramnist/data/Kircher/Kircher_GRCh38_ALL.tsv', separator='\t', infer_schema_length=10000)
variant_expr = pl.lit('chr') + pl.col('Chromosome').cast(pl.String) + pl.lit(':') + (pl.col('Position') + 1).cast(pl.String) + pl.lit(':') + pl.col('Ref').str.replace('-', '') + pl.lit('>') + pl.col('Alt').str.replace('-', '')
dataset = dataset.with_columns(variant = variant_expr)
#dataset = dataset.with_columns(variant = variant_expr).filter(pl.col('Element').__eq__('PKLR-48h') | pl.col('Element').__eq__('PKLR-24h'))
#dataset = dataset.with_columns(variant = variant_expr).filter(pl.col('Element').__eq__('F9'))

In [126]:
dataset.filter(pl.col.Element.is_in({'BCL11A', 'IRF4', 'IRF6', 'MYCrs6983267', 'MYCrs11986220', 'RET', 'SORT1', 'SORT1-flip', 'SORT1.2', 'TCF7L2', 'UC88', 'ZFAND3', 'ZRSh-13', 'ZRSh-13h2'}))

Element,Cell_Type,Chromosome,Position,Ref,Alt,Tags,DNA,RNA,Value,P-Value,variant
str,str,str,i64,str,str,i64,i64,i64,f64,f64,str
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""-""",32,577,1345,-0.34,0.00546,"""chr2:60494940:C>"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""A""",146,2785,6772,-0.05,0.38889,"""chr2:60494940:C>A"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""G""",60,975,2436,-0.13,0.13721,"""chr2:60494940:C>G"""
"""BCL11A""","""HEL92.1.7""","""2""",60494939,"""C""","""T""",1084,8543,16057,-0.7,0.0,"""chr2:60494940:C>T"""
"""BCL11A""","""HEL92.1.7""","""2""",60494940,"""C""","""A""",596,9425,23430,-0.08,0.00413,"""chr2:60494941:C>A"""
…,…,…,…,…,…,…,…,…,…,…,…
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""-""",1447,11717,34300,0.04,0.11224,"""chr7:156791603:C>"""
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""A""",25,107,309,-0.17,0.33374,"""chr7:156791603:C>A"""
"""ZRSh-13h2""","""NIH-3T3""","""7""",156791602,"""C""","""G""",654,4748,14458,0.06,0.09092,"""chr7:156791603:C>G"""


In [27]:
dataset = dataset.filter(pl.col.Tags.__gt__(10))

In [29]:
model = dna_model.create_from_huggingface('all_folds', device=jax.devices()[0], organism_settings={dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings(fasta_path='/home/jovyan/.cache/mpramnist/data/Kircher/hg38.fa'), 
                                                                                                   dna_model.Organism.MUS_MUSCULUS: (
            dna_model.OrganismSettings()
        )})

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

In [32]:
varStrs = ['chr1:209815790:G>', 'chr1:209815790:G>A']
intervals = []
variants = []
for var in varStrs:
    variant = genome.Variant.from_str(var)
    interval = variant.reference_interval.resize(2**20)
    intervals.append(interval)
    variants.append(variant)
prediction = model.score_variants(
    intervals,
    variants,
    variant_scorers=[CenterMaskScorer(requested_output=dna_model.OutputType.DNASE, width=501, aggregation_type=AggregationType.DIFF_SUM)],
    organism=dna_model.Organism.HOMO_SAPIENS,
    max_workers=20
    )

  0%|          | 0/2 [00:00<?, ?it/s]

In [118]:
ontologies = {"CL:0000312", "CL:1001606"}
df = variant_scorers.tidy_scores(prediction)[['variant_id', 'ontology_curie', 'raw_score']]
df.variant_id = df.variant_id.astype(str)

In [119]:
pl.DataFrame(df)

variant_id,ontology_curie,raw_score
str,str,f32
"""chr1:209815790:G>""","""CL:0000047""",0.332077
"""chr1:209815790:G>""","""CL:0000084""",0.325681
"""chr1:209815790:G>""","""CL:0000115""",0.346283
"""chr1:209815790:G>""","""CL:0000127""",0.370087
"""chr1:209815790:G>""","""CL:0000134""",0.060936
…,…,…
"""chr1:209815790:G>A""","""UBERON:0036149""",-0.573956
"""chr1:209815790:G>A""","""UBERON:8300001""",-0.80757
"""chr1:209815790:G>A""","""UBERON:8300002""",-0.651495


In [123]:
def select_by_ontology(struct, ontologies:set|list=["CL:0000312", "CL:1001606"]):
    ont_list = struct['ontology_curie']
    raw_scores = struct['raw_score']
    indices = [ont_list.index(ont) for ont in ontologies]
    raw_values = [raw_scores[idx] for idx in indices]
    return np.mean(raw_values) 
pl.DataFrame(df).group_by('variant_id') \
                .agg(pl.col("*")) \
                .with_columns(raw_score_averaged = pl.struct(pl.col.ontology_curie, pl.col.raw_score) \
                .map_elements(select_by_ontology, return_dtype=pl.Float64, strategy='threading')) \
                .sort(pl.col('variant_id').sort_by(pl.Series(['chr1:209815790:G>A', 'chr1:209815790:G>'])))['raw_score_averaged'].to_list()

[8.349853515625, -19.598388671875]

In [ ]:
["CL:0000047", "CL:0000084", "UBERON:8300004"]

1

In [ ]:
pl.DataFrame(df).group_by('variant_id').agg(pl.col("*")).map_rows()

In [70]:
pl.DataFrame(df.iterrows())

column_0,column_1
i64,object
0,"variant_id chr1:209815790:G> ontology_curie CL:0000047 raw_score 0.332077 Name: 0, dtype: object"
1,"variant_id chr1:209815790:G> ontology_curie CL:0000084 raw_score 0.325681 Name: 1, dtype: object"
2,"variant_id chr1:209815790:G> ontology_curie CL:0000115 raw_score 0.346283 Name: 2, dtype: object"
3,"variant_id chr1:209815790:G> ontology_curie CL:0000127 raw_score 0.370087 Name: 3, dtype: object"
4,"variant_id chr1:209815790:G> ontology_curie CL:0000134 raw_score 0.060936 Name: 4, dtype: object"
…,…
605,"variant_id chr1:209815790:G>A ontology_curie UBERON:0036149 raw_score -0.573956 Name: 605, dtype: object"
606,"variant_id chr1:209815790:G>A ontology_curie UBERON:8300001 raw_score -0.80757 Name: 606, dtype: object"
607,"variant_id chr1:209815790:G>A ontology_curie UBERON:8300002 raw_score -0.651495 Name: 607, dtype: object"


In [31]:
dataset.filter(pl.col.Element == 'IRF6')

Element,Cell_Type,Chromosome,Position,Ref,Alt,Tags,DNA,RNA,Value,P-Value,variant
str,str,str,i64,str,str,i64,i64,i64,f64,f64,str
"""IRF6""","""HaCaT""","""1""",209815789,"""G""","""-""",21,7726,6387,0.0,0.98301,"""chr1:209815790:G>"""
"""IRF6""","""HaCaT""","""1""",209815789,"""G""","""A""",108,33406,33691,-0.07,0.14117,"""chr1:209815790:G>A"""
"""IRF6""","""HaCaT""","""1""",209815789,"""G""","""C""",24,2936,5794,-0.15,0.14085,"""chr1:209815790:G>C"""
"""IRF6""","""HaCaT""","""1""",209815790,"""T""","""A""",18,3663,3857,-0.19,0.10675,"""chr1:209815791:T>A"""
"""IRF6""","""HaCaT""","""1""",209815790,"""T""","""C""",59,25296,26021,0.03,0.64083,"""chr1:209815791:T>C"""
…,…,…,…,…,…,…,…,…,…,…,…
"""IRF6""","""HaCaT""","""1""",209816386,"""A""","""G""",51,8280,7436,0.05,0.4396,"""chr1:209816387:A>G"""
"""IRF6""","""HaCaT""","""1""",209816386,"""A""","""T""",26,5570,6049,-0.16,0.10003,"""chr1:209816387:A>T"""
"""IRF6""","""HaCaT""","""1""",209816387,"""G""","""T""",46,9607,9447,-0.24,0.00089,"""chr1:209816388:G>T"""


In [4]:
predictor = AlphaGenomeScoreVariant(ontology_to_use='enformer', model = model)

In [12]:
output = predictor(variantStrings = dataset['variant'].to_list(), observed_change=dataset['Value'].to_list(), ontology="EFO:0002067")

  0%|          | 0/950 [00:00<?, ?it/s]

In [9]:
#DNase result on PKLR, filtered, tags >10
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.79778
0.79778,1.0


In [18]:
#DNase result on PKLR, filtered, tags >10, pvalue < 0.1
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.84749
0.84749,1.0


In [ ]:
#DNase result on PKLR, no filtering
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.585635
0.585635,1.0


In [13]:
#DNase result on F9, tags > 10
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.387356
0.387356,1.0


In [ ]:
#DNase result on F9
output[['Predicted', 'Observed']].corr()

Predicted,Observed
f64,f64
1.0,0.319857
0.319857,1.0
